In [15]:
import requests
import pandas as pd
import hashlib
from pathlib import Path
import gzip
import io
from datetime import time
import os

In [9]:
with open("apikey.txt", "w") as f:
    f.write("7ce6017832aa26c0268850730d85c808")
with open("apikey.txt") as f:
    apikey = f.read()

In [4]:
url = "https://api.stlouisfed.org/fred/series/observations"

In [7]:
state_map = {
    "Alabama": "ALUR",
    "Alaska": "AKUR",
    "Arizona": "AZUR",
    "Arkansas": "ARUR",
    "California": "CAUR",
    "Colorado": "COUR",
    "Connecticut": "CTUR",
    "Delaware": "DEUR",
    "Florida": "FLUR",
    "Georgia": "GAUR",
    "Hawaii": "HIUR",
    "Idaho": "IDUR",
    "Illinois": "ILUR",
    "Indiana": "INUR",
    "Iowa": "IAUR",
    "Kansas": "KSUR",
    "Kentucky": "KYUR",
    "Louisiana": "LAUR",
    "Maine": "MEUR",
    "Maryland": "MDUR",
    "Massachusetts": "MAUR",
    "Michigan": "MIUR",
    "Minnesota": "MNUR",
    "Mississippi": "MSUR",
    "Missouri": "MOUR",
    "Montana": "MTUR",
    "Nebraska": "NEUR",
    "Nevada": "NVUR",
    "New Hampshire": "NHUR",
    "New Jersey": "NJUR",
    "New Mexico": "NMUR",
    "New York": "NYUR",
    "North Carolina": "NCUR",
    "North Dakota": "NDUR",
    "Ohio": "OHUR",
    "Oklahoma": "OKUR",
    "Oregon": "ORUR",
    "Pennsylvania": "PAUR",
    "Rhode Island": "RIUR",
    "South Carolina": "SCUR",
    "South Dakota": "SDUR",
    "Tennessee": "TNUR",
    "Texas": "TXUR",
    "Utah": "UTUR",
    "Vermont": "VTUR",
    "Virginia": "VAUR",
    "Washington": "WAUR",
    "West Virginia": "WVUR",
    "Wisconsin": "WIUR",
    "Wyoming": "WYUR",
}

In [26]:
myData = []
for state, series_id in state_map.items():
    params = {
        "series_id": series_id,
        "api_key": apikey,
        "file_type": "json",
        "observation_start": "1976-01-01"
    }

    response = requests.get(url, params=params)
    data = response.json()
    observations = pd.DataFrame(data["observations"])
    observations["state"] = state
    observations["series_id"] = series_id
    myData.append(observations)

In [29]:
combined = pd.concat(myData, ignore_index=True)
combined = combined[["date", "state", "series_id", "value", "realtime_start", "realtime_end"]]
combined = combined.rename(columns={"value": "unemployment_rate"})
combined = combined.sort_values(["state", "date"]).reset_index(drop=True)
combined.to_csv("fred_unemployment_raw.csv", index=False)

In [32]:
with open("fred_unemployment_raw.csv", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
with open("fred_unemployment_raw.csv.sha", 'w') as f:
    f.write(sha256)